In [1]:
import matplotlib.pyplot as plt
import numpy as np
import os
import pandas as pd
import re
import shutil
import string
from sklearn.metrics import classification_report
import tensorflow as tf
from keras.models import Model
from tensorflow.keras import layers
from tensorflow.keras import losses
from tensorflow.keras.optimizers import Adam
from keras.callbacks import EarlyStopping,ModelCheckpoint
from tensorflow.keras.layers import Dense, Input, Dropout, Bidirectional, LSTM, Embedding, BatchNormalization,  Reshape, Conv2D, MaxPool2D, concatenate, Flatten, Activation
import torch
import numpy as np
from transformers import BertTokenizer, BertModel,RobertaTokenizer, RobertaModel
import ast

In [2]:
tf.random.set_seed(2023)

In [3]:
# Verificar si la GPU está disponible
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [4]:
archivo_3 = '/content/drive/MyDrive/UDENAR/ProyectosGrado/Errol/Ginna/DataProgramsandDescriptions-CatRetroalimentacion5000.xlsx'
train_full = pd.read_excel(archivo_3)

In [5]:
class WhileLoopFinder(ast.NodeVisitor):
    def __init__(self, source_code):
        self.source_code = source_code.splitlines()
        self.functions_with_while = []
        self.current_function = None

    def visit_FunctionDef(self, node):
        original_current_function = self.current_function
        self.current_function = {
            "function_name": node.name,
            "has_while_loop": False,
            "pre_while_code_lines": [],
            "while_loops": []
        }

        function_start_line = node.lineno - 1
        function_end_line = (node.end_lineno if hasattr(node, 'end_lineno') else len(self.source_code))
        function_lines = self.source_code[function_start_line:function_end_line]

        found_while = False
        for stmt in node.body:
            if isinstance(stmt, ast.While):
                self.current_function["has_while_loop"] = True
                found_while = True
                try:
                    while_condition = ast.unparse(stmt.test).strip()
                    print("OK condition")
                except AttributeError:
                    print("Error al obtener la condición del while")
                try:
                    while_body_code = ast.unparse(ast.Module(body=stmt.body, type_ignores=[])).strip()
                    print("Ok body")
                except AttributeError:
                    print("Error al obtener el código del cuerpo del while")

                self.current_function["while_loops"].append({
                    "condition": while_condition,
                    "body_code": while_body_code
                })
            elif not found_while:
                start_line = stmt.lineno - 1
                end_line = (stmt.end_lineno if hasattr(stmt, 'end_lineno') else stmt.lineno)
                self.current_function["pre_while_code_lines"].extend(self.source_code[start_line:end_line])
            self.generic_visit(stmt)

        if self.current_function["has_while_loop"]:
            last_while_end_line = None
            if self.current_function["while_loops"]:
                last_while_node = next((node for node in reversed(node.body) if isinstance(node, ast.While)), None)
                if last_while_node and hasattr(last_while_node, 'end_lineno'):
                    last_while_end_line = last_while_node.end_lineno
            post_while_code_lines = []
            if last_while_end_line:
                function_indent = len(self.source_code[node.lineno - 1]) - len(self.source_code[node.lineno - 1].lstrip())
                for line in self.source_code[last_while_end_line:function_end_line]:
                    if line.startswith(self.source_code[node.lineno - 1][:function_indent] + "    "):
                        post_while_code_lines.append(line[function_indent + 4:])
                    else:
                        post_while_code_lines.append(line[function_indent:])
            self.current_function["post_while_code_lines"] = "\n".join(post_while_code_lines).strip()
            self.current_function["pre_while_code_lines"] = "\n".join(self.current_function["pre_while_code_lines"]).strip()
            self.functions_with_while.append(self.current_function)

        self.current_function = original_current_function



In [6]:
def DGries_states(solution):
  try:
    # Crear el AST
    tree = ast.parse(solution)
    # Recorrer el AST con nuestro visitante
    finder = WhileLoopFinder(solution)
    finder.visit(tree)
    for func_info in finder.functions_with_while:
      initial_state=func_info["pre_while_code_lines"] if func_info["pre_while_code_lines"] else False
      end_state=""
      transformation_state=""
      for j, wl in enumerate(func_info["while_loops"]):
          end_state=wl["condition"]
          transformation_state=wl["body_code"]

      if not initial_state:
        initial_state=transformation_state
      return initial_state,transformation_state,end_state
  except Exception as error:
    return "Exception:"+str(error),"Exception:"+str(error),"Exception:"+str(error)




In [7]:
class EncodeTextSource:
  def __init__(self):
    self.is_loadtokenizers=False



  def tokenize_and_generate_embeddings_descriptions(self,description):
    if not self.is_loadtokenizers:
      raise ValueError("tokenizers dont load")
    # Tokenize input description and move tokens to GPU
    tokens = self.beart_tokenizer.encode_plus(description, return_tensors="pt", truncation=True)
    tokens = {key: value.to(device) for key, value in tokens.items()}
    # Generate embeddings
    with torch.no_grad():
        outputs = self.beart_model(**tokens)
    # Extract embeddings for all tokens
    desc_embeddings = outputs.last_hidden_state.cpu().numpy()
    return desc_embeddings

  def tokenize_and_generate_embeddings_codes(self,code):
    if not self.is_loadtokenizers:
      raise ValueError("tokenizers dont load")
    # Tokenize input code and move tokens to GPU
    tokens = self.beart_tokenizer.encode_plus(code, return_tensors="pt", truncation=True)
    tokens = {key: value.to(device) for key, value in tokens.items()}
    # Generate embeddings
    with torch.no_grad():
        outputs = self.codebeart_model(**tokens)
    # Extract embeddings for all tokens
    code_embeddings = outputs.last_hidden_state.cpu().numpy()

    return code_embeddings

  def load_beart_tokenizer(self):
    # Verificar si la GPU está disponible
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    # Cargar el tokenizer y el modelo en la GPU si está disponible
    model_name = "bert-base-uncased"
    self.beart_model = BertModel.from_pretrained(model_name)
    self.beart_tokenizer = BertTokenizer.from_pretrained(model_name)
    self.beart_model.to(device)

  def load_codebert_tokenizer(self):
    # Verificar si la GPU está disponible
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    # Cargar el tokenizer y el modelo en la GPU si está disponible
    self.codebeart_tokenizer = RobertaTokenizer.from_pretrained("microsoft/codebert-base")
    self.codebeart_model = RobertaModel.from_pretrained("microsoft/codebert-base")
    self.codebeart_model.to(device)

  def start_tokenizer(self):
    self.load_beart_tokenizer()
    self.load_codebert_tokenizer()
    self.is_loadtokenizers=True




# All characteristics

In [8]:
def categoricallabelAll(w):

  if w=="['Initial state']":
    return 0
  if w=="['Final state']":
    return 1
  if w=="['State transformation']":
    return 2
  if w=="['Initial state', 'Final state']":
    return 3
  if w=="['Initial state', 'State transformation']":
    return 4
  if w=="['Final state', 'State transformation']":
    return 5
  if w=="['Initial state', 'Final state', 'State transformation']":
    return 6
  return 7

category=np.array([
    'Initial state',
    'Final state',
    'State transformation',
    'Initial state, Final state',
    'Initial state, State transformation',
    'Final state, State transformation',
    'Initial state, Final state, State transformation'

])

# Load Dataset

In [9]:
train_full.head()

,No.,Problema,Solución,Estado incial,Estado final,Transformación de estado,Etiqueta 1,Etiqueta 2,Realimentación
0,1,Write a Python function that returns the facto...,def factorial(n):\n result = 1\n i = 1\n...,"result = 1, i = 1",i <= n,"result *= i, i += 1",Correct,['Correct'],NaN
1,2,Write a Python function that returns the sum o...,def sum_1_to_100():\n total = 0\n i = 1\...,"total = 0, i = 1, n = 100",i <= n,"total += i, i += 1",Correct,['Correct'],NaN
2,3,Write a Python function that prints the number...,def print_and_store():\n numbers = []\n ...,"numbers = [], i = 0, n = 10",i <= n,"print(i), numbers.append(i), i += 1",Correct,['Correct'],NaN
3,4,Write a Python function to print the numbers f...,def print_and_store_reverse():\n numbers = ...,"numbers = [], i = 10, n = 1",i >= n,"print(i), numbers.append(i), i -= 1",Correct,['Correct'],NaN
4,5,Write a Python function to check if a number i...,def is_prime(num):\n if num <= 1:\n ...,"i = 2, i = = 0:",i <= num//2,"if num % i == 0:, return False, i += 1",Correct,['Correct'],NaN


In [10]:
train_full.drop(["No.","Realimentación","Estado incial","Estado final","Transformación de estado"],axis=1,inplace=True)
train_full

,Problema,Solución,Etiqueta 1,Etiqueta 2
0,Write a Python function that returns the facto...,def factorial(n):\n result = 1\n i = 1\n...,Correct,['Correct']
1,Write a Python function that returns the sum o...,def sum_1_to_100():\n total = 0\n i = 1\...,Correct,['Correct']
2,Write a Python function that prints the number...,def print_and_store():\n numbers = []\n ...,Correct,['Correct']
3,Write a Python function to print the numbers f...,def print_and_store_reverse():\n numbers = ...,Correct,['Correct']
4,Write a Python function to check if a number i...,def is_prime(num):\n if num <= 1:\n ...,Correct,['Correct']
...,...,...,...,...
4995,Write a Python function that returns the sum o...,def sum_of_first_five_numbers():\n number =...,Incorrect,"['Initial state', 'Final state']"
4996,Write a Python function that returns the facto...,def factorial(n):\n result = 0\n i = 0\n...,Incorrect,"['Initial state', 'Final state']"
4997,Write a Python function that prints the number...,def print_and_store_numbers():\n numbers = ...,Incorrect,"['Initial state', 'Final state']"
4998,Write a Python function to print the numbers f...,def print_and_store_numbers():\n numbers = ...,Incorrect,"['Initial state', 'Final state']"


In [11]:
train_full[["Estado incial","Transformación de estado","Estado final"]]=train_full['Solución'].apply(DGries_states).apply(pd.Series)

Streaming output truncated to the last 5000 lines.
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK c

In [12]:
train_full = train_full[train_full['Etiqueta 1']!='Correct'].copy()

In [13]:
y=train_full['Etiqueta 2'].apply(categoricallabelAll)
np.unique(y)

array([0, 1, 2, 3, 4, 5, 6])

In [14]:
train_full.dropna(inplace=True)

In [15]:
np.unique(train_full['Etiqueta 2'])

array(["['Final state', 'State transformation']", "['Final state']",
       "['Initial state', 'Final state', 'State transformation']",
       "['Initial state', 'Final state']",
       "['Initial state', 'State transformation']", "['Initial state']",
       "['State transformation']"], dtype=object)

In [16]:
encoder=EncodeTextSource()
encoder.start_tokenizer()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

...

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/150 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/498 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/499M [00:00<?, ?B/s]

In [17]:
%%time
problem=train_full['Problema'].apply(encoder.tokenize_and_generate_embeddings_descriptions).to_numpy()
startstate = train_full['Estado incial'].apply(encoder.tokenize_and_generate_embeddings_codes).to_numpy()
finalstate = train_full['Estado final'].apply(encoder.tokenize_and_generate_embeddings_codes).to_numpy()
transstate = train_full['Transformación de estado'].apply(encoder.tokenize_and_generate_embeddings_codes).to_numpy()

model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

CPU times: user 2min 6s, sys: 2.5 s, total: 2min 9s
Wall time: 2min 28s


In [62]:
problem.shape,startstate.shape,finalstate.shape,transstate.shape

((3459,), (3459,), (3459,), (3459,))

In [63]:
print(startstate.shape)
print(startstate[1262].shape)


(3459,)
(1, 9, 768)


In [64]:
Xp=np.array([sentence[0].mean(axis=0) for sentence in problem])
Xs=np.array([sentence[0].mean(axis=0) for sentence in startstate])
Xf=np.array([sentence[0].mean(axis=0) for sentence in finalstate])
Xt=np.array([sentence[0].mean(axis=0) for sentence in transstate])
Xp.shape,Xs.shape,Xf.shape,Xt.shape

((3459, 768), (3459, 768), (3459, 768), (3459, 768))

In [65]:
y=train_full['Etiqueta 2'].apply(categoricallabelAll)
y=y.to_numpy()
np.unique(y)

array([0, 1, 2, 3, 4, 5, 6])

In [66]:
from sklearn.model_selection import train_test_split

Xp_train,Xp_test,Xs_train,Xs_test,Xt_train,Xt_test,Xf_train,Xf_test,y_train,y_test=train_test_split(Xp,Xs,Xt,Xf,y,test_size=0.2,random_state=2023, stratify=y)

In [67]:
Xp_train.shape,Xp_test.shape,Xs_train.shape,Xs_test.shape,Xt_train.shape,Xt_test.shape,Xf_train.shape,Xf_test.shape,y_train.shape,y_test.shape

((2767, 768),
 (692, 768),
 (2767, 768),
 (692, 768),
 (2767, 768),
 (692, 768),
 (2767, 768),
 (692, 768),
 (2767,),
 (692,))

# Keras model

In [68]:
from tensorflow.keras import backend as K
import gc
def ANNTC(input,base,pow_initial,num_max_blocks):

  drop_out=0.5
  print("base",base,"pow",pow_initial,"num_max_blocks",num_max_blocks)
  n=num_max_blocks//2
  neurons=int(base**(pow_initial+n+1))
  # Encoder
  x=None
  print("Encoder",n)
  try:
    for i in range(n):
      x=Dense(neurons)(input)
      x=BatchNormalization()(x)
      x=Activation('relu')(x)
      x=Dropout(drop_out)(x)
      input=x
      drop_out=0.2
      print("Block ",i,neurons)
      neurons=int(neurons/base)



    #BottleNeck
    x=Dense(neurons)(x)
    x=BatchNormalization()(x)
    x=Activation('relu')(x)
    x=Dropout(0.2)(x)
    print("BottleNeck",neurons)

    # Decoder
    print("Decoder",n)
    for i in range(n):
      neurons=int(neurons*base)
      print("Block ",i,neurons)
      x=Dense(neurons)(x)
      x=BatchNormalization()(x)
      x=Activation('relu')(x)
      x=Dropout(drop_out)(x)
      if i==n-1:
        drop_out=0.5
      drop_out=0.2
  except Exception as err:

    K.clear_session()
    gc.collect()
    del x
    print(f"Unexpected {err=}, {type(err)=}")
    raise

  return x


def neural_network(problem,start_input,trass_input,final_input,base=10,pow_initial=4,num_max_blocks=3):
  mlp_p=ANNTC(problem,base,pow_initial,num_max_blocks)
  mlp_s=ANNTC(start_input,base,pow_initial,num_max_blocks)
  mlp_t=ANNTC(trass_input,base,pow_initial,num_max_blocks)
  mlp_f=ANNTC(final_input,base,pow_initial,num_max_blocks)
  ## Concatenate All models
  combined = concatenate([mlp_p, mlp_s, mlp_t,mlp_f])
  ## output layer
  output=Dense(8)(combined)
  output=BatchNormalization()(output)
  output=Activation('softmax')(output)
  ## Model
  model = Model(inputs=[problem_input, start_input, trass_input,final_input], outputs=output)
  return model

In [69]:
np.logspace(2, 4, num=3, base=10)


array([  100.,  1000., 10000.])

In [70]:
dftimes=pd.read_csv("/content/drive/MyDrive/UDENAR/ProyectosGrado/Errol/Ginna/times_tuning.csv",usecols=['tuning','times'])


In [71]:
dftimes['tuning']=dftimes['tuning'].apply(lambda x: x.replace(",","_"))
dftimes

,tuning,times
0,3_2_0,67.592388
1,3_2_1,55.045458
2,3_2_2,66.796277
3,3_2_3,73.405371
4,3_8_0,69.524091
5,3_8_1,69.826996
6,3_8_2,119.134001
7,3_10_0,67.938086
8,3_10_1,76.453464
9,3_10_2,327.824992


In [72]:
max_val_acuracies=[]
max_accuracies=[]
configurations=[]
# read histories csv
for num_max_blocks in [3,5,7]:
  print("Bloque")
  for base in [2,8,10,16]:
    print("")
    for pow_initial in [0,1,2,3]:
      try:
        df_histories=pd.read_csv('/content/drive/MyDrive/UDENAR/ProyectosGrado/Errol/Ginna/histories_{0}.csv'.format(f"{num_max_blocks}_{base}_{pow_initial}"))
        configurations.append(f"{num_max_blocks}_{base}_{pow_initial}")
        max_val_acuracies.append(max(df_histories['val_accuracy']))
        max_accuracies.append(max(df_histories['accuracy']))
        print(f"{num_max_blocks}_{base}_{pow_initial}")
        print("max_val_accuracy",max(df_histories['val_accuracy']))
        print("max_accuracy",max(df_histories['accuracy']))
      except:
        print("Error")

Bloque

3_2_0
max_val_accuracy 0.7129963636398315
max_accuracy 0.6127429008483887
3_2_1
max_val_accuracy 0.7870036363601685
max_accuracy 0.7478535771369934
3_2_2
max_val_accuracy 0.8050541281700134
max_accuracy 0.8120198845863342
3_2_3
max_val_accuracy 0.907942235469818
max_accuracy 0.9389968514442444

3_8_0
max_val_accuracy 0.8880866169929504
max_accuracy 0.91640305519104
3_8_1
max_val_accuracy 0.9386281371116638
max_accuracy 0.9972887635231018
3_8_2
max_val_accuracy 0.9332129955291748
max_accuracy 1.0
Error

3_10_0
max_val_accuracy 0.9277978539466858
max_accuracy 0.9624943733215332
3_10_1
max_val_accuracy 0.9404332041740416
max_accuracy 0.9995481371879578
3_10_2
max_val_accuracy 0.92418771982193
max_accuracy 1.0
Error

3_16_0
max_val_accuracy 0.9386281371116638
max_accuracy 0.9868956208229064
3_16_1
max_val_accuracy 0.929602861404419
max_accuracy 1.0
Error
Error
Bloque

5_2_0
max_val_accuracy 0.5685920715332031
max_accuracy 0.4979665577411651
5_2_1
max_val_accuracy 0.7924187779426575

In [73]:
dfall=pd.DataFrame({"configurations":configurations,"max_val_acuracies":max_val_acuracies,"max_accuracies":max_accuracies})
dfall

,configurations,max_val_acuracies,max_accuracies
0,3_2_0,0.712996,0.612743
1,3_2_1,0.787004,0.747854
2,3_2_2,0.805054,0.812020
3,3_2_3,0.907942,0.938997
4,3_8_0,0.888087,0.916403
5,3_8_1,0.938628,0.997289
6,3_8_2,0.933213,1.000000
7,3_10_0,0.927798,0.962494
8,3_10_1,0.940433,0.999548
9,3_10_2,0.924188,1.000000


In [74]:
df_merged = pd.merge(dfall, dftimes, left_on='configurations', right_on='tuning')
df_merged

,configurations,max_val_acuracies,max_accuracies,tuning,times
0,3_2_0,0.712996,0.612743,3_2_0,67.592388
1,3_2_1,0.787004,0.747854,3_2_1,55.045458
2,3_2_2,0.805054,0.812020,3_2_2,66.796277
3,3_2_3,0.907942,0.938997,3_2_3,73.405371
4,3_8_0,0.888087,0.916403,3_8_0,69.524091
5,3_8_1,0.938628,0.997289,3_8_1,69.826996
6,3_8_2,0.933213,1.000000,3_8_2,119.134001
7,3_10_0,0.927798,0.962494,3_10_0,67.938086
8,3_10_1,0.940433,0.999548,3_10_1,76.453464
9,3_10_2,0.924188,1.000000,3_10_2,327.824992


In [75]:
from sklearn.metrics import matthews_corrcoef, precision_recall_curve, auc
from sklearn.preprocessing import label_binarize
import numpy as np

def calculate_mcc_multiclass(y_true, y_pred_probs):
    y_pred_labels = np.argmax(y_pred_probs, axis=1)
    # Convertir etiquetas verdaderas one-hot a etiquetas enteras si es necesario
    #y_true_labels = np.argmax(y_true, axis=1) if y_true.ndim > 1 else y_true
    return matthews_corrcoef(y_true, y_pred_labels)

def calculate_auc_pr_multiclass(y_true, y_pred_probs, average='macro'):
    n_classes = y_pred_probs.shape[1]
    #y_true_labels = np.argmax(y_true, axis=1) if y_true.ndim > 1 else y_true
    y_true_bin = label_binarize(y_true, classes=range(n_classes))

    auc_pr_list = []
    for i in range(n_classes):
        precision, recall, _ = precision_recall_curve(y_true_bin[:, i], y_pred_probs[:, i])
        auc_pr_list.append(auc(recall, precision))

    if average == 'macro':
        return np.mean(auc_pr_list)
    elif average == 'weighted':
        class_counts = y_true_bin.sum(axis=0)
        return np.average(auc_pr_list, weights=class_counts)
    else:
        return auc_pr_list


In [76]:
from time import time
from tensorflow import keras
times_predict=[]
accuracies_predict=[]
loss_predict=[]
mccs_predict=[]
aucpr_predict=[]
for num_max_blocks in [3,5,7]:
  print("Bloque")
  for base in [2,8,10,16]:
    print("")
    for pow_initial in [0,1,2,3]:
      try:
        model=keras.models.load_model('/content/drive/MyDrive/UDENAR/ProyectosGrado/Errol/Ginna/best_finall_{0}.keras'.format(f"{num_max_blocks}_{base}_{pow_initial}"))

        evaluate=model.evaluate([Xp_test,Xs_test,Xt_test,Xf_test],y_test)


        t1=time()
        y_pred=model.predict([Xp_test,Xs_test,Xt_test,Xf_test])
        t2=time()
        mcc=calculate_mcc_multiclass(y_test, y_pred)
        auc_pr=calculate_auc_pr_multiclass(y_test, y_pred)

        times_predict.append(t2-t1)
        accuracies_predict.append(evaluate[1])
        loss_predict.append(evaluate[0])
        mccs_predict.append(mcc)
        aucpr_predict.append(auc_pr)


        accuracy=evaluate[1]
        loss=evaluate[0]

        print(f"{num_max_blocks}_{base}_{pow_initial}")
        print("accuracy",accuracy)
        print("loss",loss)
        print("mcc",mcc)
        print("auc_pr",auc_pr)
        print("time predict",t2-t1)
      except:
        print("Error")


Bloque

22/22 ━━━━━━━━━━━━━━━━━━━━ 2s 29ms/step - accuracy: 0.7118 - loss: 1.0682
22/22 ━━━━━━━━━━━━━━━━━━━━ 2s 40ms/step


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_ranking.py:1033: UserWarning: No positive class found in y_true, recall is set to one for all thresholds.
  warnings.warn(


3_2_0
accuracy 0.6907514333724976
loss 1.080087661743164
mcc 0.5736442610297167
auc_pr 0.5501402185814949
time predict 2.610106945037842
22/22 ━━━━━━━━━━━━━━━━━━━━ 2s 29ms/step - accuracy: 0.7858 - loss: 0.7809
22/22 ━━━━━━━━━━━━━━━━━━━━ 2s 37ms/step


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_ranking.py:1033: UserWarning: No positive class found in y_true, recall is set to one for all thresholds.
  warnings.warn(


3_2_1
accuracy 0.7716763019561768
loss 0.7936642169952393
mcc 0.6889332308488916
auc_pr 0.6325096673483722
time predict 2.613360643386841
22/22 ━━━━━━━━━━━━━━━━━━━━ 2s 28ms/step - accuracy: 0.8202 - loss: 0.7111
22/22 ━━━━━━━━━━━━━━━━━━━━ 3s 83ms/step
3_2_2
accuracy 0.8034682273864746
loss 0.735679566860199
mcc 0.7332125949163534
auc_pr 0.6827054462709273
time predict 2.6083309650421143


...

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_ranking.py:1033: UserWarning: No positive class found in y_true, recall is set to one for all thresholds.
  warnings.warn(


22/22 ━━━━━━━━━━━━━━━━━━━━ 3s 47ms/step - accuracy: 0.8563 - loss: 0.4135
22/22 ━━━━━━━━━━━━━━━━━━━━ 3s 73ms/step


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_ranking.py:1033: UserWarning: No positive class found in y_true, recall is set to one for all thresholds.
  warnings.warn(


7_8_0
accuracy 0.8728323578834534
loss 0.41608425974845886
mcc 0.8298959519834477
auc_pr 0.8500716811460107
time predict 5.168895959854126
Error
Error
Error

22/22 ━━━━━━━━━━━━━━━━━━━━ 5s 51ms/step - accuracy: 0.8891 - loss: 0.3847
22/22 ━━━━━━━━━━━━━━━━━━━━ 3s 78ms/step
7_10_0
accuracy 0.8887283205986023
loss 0.40323254466056824
mcc 0.8513848504480642
auc_pr 0.8690009642440474
time predict 5.171505928039551
Error
Error
Error

Error
Error
Error
Error


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_ranking.py:1033: UserWarning: No positive class found in y_true, recall is set to one for all thresholds.
  warnings.warn(


In [77]:
df_merged['predict_time']=times_predict
df_merged['predict_accuracy']=accuracies_predict
df_merged['predict_loss']=loss_predict
df_merged

,configurations,max_val_acuracies,max_accuracies,tuning,times,predict_time,predict_accuracy,predict_loss
0,3_2_0,0.712996,0.612743,3_2_0,67.592388,2.610107,0.690751,1.080088
1,3_2_1,0.787004,0.747854,3_2_1,55.045458,2.613361,0.771676,0.793664
2,3_2_2,0.805054,0.812020,3_2_2,66.796277,2.608331,0.803468,0.735680
3,3_2_3,0.907942,0.938997,3_2_3,73.405371,2.650538,0.881503,0.377821
4,3_8_0,0.888087,0.916403,3_8_0,69.524091,2.609542,0.867052,0.462008
5,3_8_1,0.938628,0.997289,3_8_1,69.826996,1.731248,0.923410,0.286407
6,3_8_2,0.933213,1.000000,3_8_2,119.134001,1.681532,0.911850,0.311886
7,3_10_0,0.927798,0.962494,3_10_0,67.938086,2.608639,0.907514,0.330063
8,3_10_1,0.940433,0.999548,3_10_1,76.453464,1.576468,0.920520,0.329267
9,3_10_2,0.924188,1.000000,3_10_2,327.824992,2.614919,0.903179,0.345134


In [82]:
df_merged['predict_mcc']=mccs_predict
df_merged['predict_aucpr']=aucpr_predict

In [83]:
df_merged

,configurations,max_val_acuracies,max_accuracies,tuning,times,predict_time,predict_accuracy,predict_loss,predict_mcc,predict_aucpr
0,3_2_0,0.712996,0.612743,3_2_0,67.592388,2.610107,0.690751,1.080088,0.573644,0.550140
1,3_2_1,0.787004,0.747854,3_2_1,55.045458,2.613361,0.771676,0.793664,0.688933,0.632510
2,3_2_2,0.805054,0.812020,3_2_2,66.796277,2.608331,0.803468,0.735680,0.733213,0.682705
3,3_2_3,0.907942,0.938997,3_2_3,73.405371,2.650538,0.881503,0.377821,0.841207,0.854185
4,3_8_0,0.888087,0.916403,3_8_0,69.524091,2.609542,0.867052,0.462008,0.820862,0.835767
5,3_8_1,0.938628,0.997289,3_8_1,69.826996,1.731248,0.923410,0.286407,0.897806,0.886603
6,3_8_2,0.933213,1.000000,3_8_2,119.134001,1.681532,0.911850,0.311886,0.882640,0.885253
7,3_10_0,0.927798,0.962494,3_10_0,67.938086,2.608639,0.907514,0.330063,0.876141,0.878156
8,3_10_1,0.940433,0.999548,3_10_1,76.453464,1.576468,0.920520,0.329267,0.893882,0.883061
9,3_10_2,0.924188,1.000000,3_10_2,327.824992,2.614919,0.903179,0.345134,0.871070,0.882775


In [78]:
df_merged.to_csv("/content/drive/MyDrive/UDENAR/ProyectosGrado/Errol/Ginna/fine_tuning.csv",index=False)

In [92]:
df_merged[df_merged['predict_aucpr']>0.88]

,configurations,max_val_acuracies,max_accuracies,tuning,times,predict_time,predict_accuracy,predict_loss,predict_mcc,predict_aucpr
5,3_8_1,0.938628,0.997289,3_8_1,69.826996,1.731248,0.923410,0.286407,0.897806,0.886603
6,3_8_2,0.933213,1.000000,3_8_2,119.134001,1.681532,0.911850,0.311886,0.882640,0.885253
8,3_10_1,0.940433,0.999548,3_10_1,76.453464,1.576468,0.920520,0.329267,0.893882,0.883061
9,3_10_2,0.924188,1.000000,3_10_2,327.824992,2.614919,0.903179,0.345134,0.871070,0.882775
10,3_16_0,0.938628,0.986896,3_16_0,78.348986,2.609993,0.916185,0.285056,0.888311,0.889212


In [98]:
df_merged[(df_merged['predict_aucpr']>0.88)&
          (df_merged['max_accuracies']-df_merged['max_val_acuracies']<0.059)]

,configurations,max_val_acuracies,max_accuracies,tuning,times,predict_time,predict_accuracy,predict_loss,predict_mcc,predict_aucpr
5,3_8_1,0.938628,0.997289,3_8_1,69.826996,1.731248,0.923410,0.286407,0.897806,0.886603
10,3_16_0,0.938628,0.986896,3_16_0,78.348986,2.609993,0.916185,0.285056,0.888311,0.889212
